# Reddit Data Puller

This notebook pulls posts and comments from multiple Reddit subreddits based on keywords and saves them to CSV.

In [ ]:
# Import required libraries
import praw  # Reddit API wrapper
from prawcore import TooManyRequests  # Rate limit handling
from datetime import datetime
import time
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## Configuration

Initialize Reddit API credentials and helper functions.

In [ ]:
# Initialize Reddit API client (credentials should be set via environment variables or config)
reddit = praw.Reddit(
    user_agent=True,
    client_id="",
    client_secret="",
    username="",
    password=""
)
print("Reddit client initialized")

In [ ]:
# Helper function: Convert Unix timestamp to readable datetime
def unix_to_datetime(unix_time):
    return datetime.fromtimestamp(unix_time).strftime('%Y-%m-%d %H:%M:%S')

## Save Function

Utility function to export DataFrame to CSV.

## Data Pulling Function

Main function to pull posts and comments from Reddit based on specified criteria.

In [ ]:
def pull_reddit_data(
    reddit,
    subreddit_names,
    keywords,
    min_upvotes,
    start_date,
    end_date,
    min_comment_upvotes,
    max_posts=5000
):
    """
    Pull posts and comments from multiple Reddit subreddits based on criteria and store in DataFrame.
    
    Parameters:
    - reddit: PRAW Reddit instance
    - subreddit_names: List of subreddit names to search
    - keywords: List of keywords to search for in posts and comments
    - min_upvotes: Minimum upvotes for posts (can be None to ignore)
    - start_date: Unix timestamp for earliest post
    - end_date: Unix timestamp for latest post
    - min_comment_upvotes: Minimum upvotes for comments (can be None to ignore)
    - max_posts: Maximum number of posts to retrieve per subreddit
    
    Returns:
    - DataFrame containing posts and comments data
    """
    # Initialize list to store data
    data = []
    
    # Convert keywords to lowercase for case-insensitive search
    keywords = [k.lower() for k in keywords]
    
    # Helper function to safely get author karma
    def get_author_karma(author):
        try:
            if author and hasattr(author, 'link_karma') and hasattr(author, 'comment_karma'):
                return author.link_karma, author.comment_karma
            else:
                return 0, 0
        except Exception:
            return 0, 0
    
    # Iterate over each subreddit
    for subreddit_name in subreddit_names:
        retry_count = 0
        max_retries = 3
        while retry_count < max_retries:
            try:
                subreddit = reddit.subreddit(subreddit_name)
                post_count = 0
                comment_count = 0
                
                # Search posts
                for submission in subreddit.search(
                    ' '.join(keywords),
                    sort='relevance',
                    time_filter='all',
                    limit=max_posts
                ):
                    # Check post criteria
                    post_time = submission.created_utc
                    upvote_check = min_upvotes is None or submission.ups >= min_upvotes
                    
                    if (upvote_check and start_date <= post_time <= end_date):
                        
                        # Get author karma safely
                        author_link_karma, author_comment_karma = get_author_karma(submission.author)
                        
                        # Store post data
                        post_data = {
                            'type': 'post',
                            'id': submission.id,
                            'title': submission.title,
                            'text': submission.selftext,
                            'url': submission.url,
                            'upvotes': submission.ups,
                            'created_utc': unix_to_datetime(post_time),
                            'author': submission.author.name if submission.author else '[deleted]',
                            'subreddit': submission.subreddit.display_name,
                            'author_link_karma': author_link_karma,
                            'author_comment_karma': author_comment_karma
                        }
                        data.append(post_data)
                        post_count += 1
                        
                        # Get comments (limited to 1000 more comments)
                        submission.comments.replace_more(limit=1000)
                        for comment in submission.comments.list():
                            comment_upvote_check = min_comment_upvotes is None or comment.ups >= min_comment_upvotes
                            
                            if (comment_upvote_check and
                                any(keyword in comment.body.lower() for keyword in keywords)):
                                
                                # Get comment author karma safely
                                comment_author_link_karma, comment_author_comment_karma = get_author_karma(comment.author)
                                
                                comment_data = {
                                    'type': 'comment',
                                    'id': comment.id,
                                    'title': '',
                                    'text': comment.body,
                                    'url': f'https://reddit.com{comment.permalink}',
                                    'upvotes': comment.ups,
                                    'created_utc': unix_to_datetime(comment.created_utc),
                                    'author': comment.author.name if comment.author else '[deleted]',
                                    'subreddit': submission.subreddit.display_name,
                                    'parent_post_id': submission.id,
                                    'author_link_karma': comment_author_link_karma,
                                    'author_comment_karma': comment_author_comment_karma
                                }
                                data.append(comment_data)
                                comment_count += 1
                        
                        # Respect Reddit API rate limits
                        time.sleep(1)
                
                # Log progress
                print(f"Processed {subreddit_name}: {post_count} posts, {comment_count} comments")
                break
                
            except TooManyRequests:
                retry_count += 1
                print(f"Rate limit hit for subreddit {subreddit_name} (retry {retry_count}/{max_retries}). Waiting 120 seconds...")
                time.sleep(120)
                continue
            except Exception as e:
                print(f"Error processing subreddit {subreddit_name}: {e}")
                break
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Reorder columns
    columns = [
        'type', 'id', 'parent_post_id', 'subreddit', 'title', 'text',
        'url', 'upvotes', 'created_utc', 'author', 'author_link_karma', 'author_comment_karma'
    ]
    # Ensure all columns exist
    for col in columns:
        if col not in df.columns:
            df[col] = ''
    
    df = df[columns]
    
    return df

## Pull ETH Data

Fetch Reddit posts and comments related to Ethereum (ETH) from finance-focused subreddits.

In [ ]:
if __name__ == "__main__":
    # Example parameters - finance-focused subreddits for Ethereum
    subreddit_names = [
        "Finance",
        "Stocks",
        "Bitcoin",
        "SecurityAnalysis",
        "Wallstreetbetsnew",
        "StocksAndTrading",
        "PennyStocks",
        "algotrading",
        "babystreetbets",
        "Economics",
        "ASX_Bets",
        "antstreetbets",
        "quant",
        "weedstocks",
        "investing",
        "Economy",
        "shortinterestbets",
        "thetagang",
        "Pennystocks",
        "InvestingRetards",
        "wallstreetbets",
        "wallstreetbetsoptions",
        "wallstreetbetsELITE",
        "wallstreetbetsGER",
        "econmonitor",
        "Wallstreetwarrior",
        "StockMarket",
        "wallstreetbets2",
        "Trading",
        "WSBAfterHours",
        "smallstreetbets",
        "retardbets",
        "InvestmentClub",
        "IndianStreetBets",
        "wallstreetsidebets",
        "baystreetbets",
        "ameisenstrassenwetten",
        "wallstreetbets_",
        "ISKbets",
        "quantfinance",
        "stonks",
        "GlobalMarkets",
        "Daytrading",
        "RobinHoodPennyStocks",
        "CanadianInvestor",
        "Options",
        "MoonBets",
        "farialimabets",
        "Wallstreetsilver",
        "Cryptocurrency",
        "UKInvesting",
        "ausstocks",
        "dividends",
        "ValueInvesting",
        "financialindependence",
        "DeepFuckingValue",
        "UndervaluedStonks",
        "investing_discussion",
        "StockAnalysis",
        "FinancialPlanning",
        "PersonalFinance",
        "Fire",
        "leanfire",
        "fatFIRE",
        "UKPersonalFinance",
        "eupersonalfinance",
        "PersonalFinanceCanada",
        "AusFinance",
        "IndiaInvestments",
        "GermanFI",
        "ItaliaPersonalFinance"
    ]

    keywords = ["Ethereum", "ETH"]
    min_upvotes = 5  # Minimum post upvotes
    min_comment_upvotes = 2  # Minimum comment upvotes
    start_date = int(datetime(2025, 3, 1).timestamp())  
    end_date = int(datetime(2025, 7, 8).timestamp())  
    output_file = 'ETH'
    
    # Pull data
    df = pull_reddit_data(
        reddit=reddit,
        subreddit_names=subreddit_names,
        keywords=keywords,
        min_upvotes=min_upvotes,
        start_date=start_date,
        end_date=end_date,
        min_comment_upvotes=min_comment_upvotes
    )

    # Save to CSV
    save_to_csv(df, output_file)

Processed Finance: 0 posts, 0 comments
Processed Stocks: 1 posts, 4 comments
Processed Bitcoin: 3 posts, 2 comments
Processed SecurityAnalysis: 0 posts, 0 comments
Processed Wallstreetbetsnew: 0 posts, 0 comments
Processed StocksAndTrading: 0 posts, 0 comments
Processed PennyStocks: 1 posts, 0 comments
Processed algotrading: 0 posts, 0 comments
Processed babystreetbets: 0 posts, 0 comments
Processed Economics: 0 posts, 0 comments
Processed ASX_Bets: 0 posts, 0 comments
Processed antstreetbets: 0 posts, 0 comments
Processed quant: 0 posts, 0 comments
Processed weedstocks: 0 posts, 0 comments
Processed investing: 0 posts, 0 comments
Processed Economy: 0 posts, 0 comments
Processed shortinterestbets: 0 posts, 0 comments
Processed thetagang: 0 posts, 0 comments
Processed Pennystocks: 1 posts, 0 comments
Processed InvestingRetards: 0 posts, 0 comments
Processed wallstreetbets: 2 posts, 5 comments
Processed wallstreetbetsoptions: 0 posts, 0 comments
Processed wallstreetbetsELITE: 0 posts, 0 

In [ ]:
def save_to_csv(df, filename):
    """Save DataFrame to CSV file."""
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Data saved to {filename}")

## Pull GitLab Data

Fetch Reddit posts and comments related to GitLab (software development tool) from tech-focused subreddits.

In [ ]:
if __name__ == "__main__":
    # Tech/Programming/DevOps focused subreddits - GitLab as a product/tool
    subreddit_names = [
        # Core Programming & Development
        "programming",
        "webdev",
        "Frontend",
        "Backend",
        "FullStack",
        "learnprogramming",
        "experienceddevs",
        "softwaredevelopment",
        "softwareengineering",
        "coding",
        "compsci",
        "ProgrammerHumor",
        "cscareerquestions",
        "ITCareerQuestions",
        "developers",
        "webdevelopment",
        "javascript",
        "Python",
        "node",
        "reactjs",
        "angular",
        "vuejs",
        "php",
        "dotnet",
        "java",
        "golang",
        "rust",
        "ruby",
        "cplusplus",
        
        # DevOps & Infrastructure
        "devops",
        "sysadmin",
        "linuxadmin",
        "kubernetes",
        "docker",
        "containerization",
        "microservices",
        "cloudcomputing",
        "aws",
        "azure",
        "googlecloud",
        "terraform",
        "ansible",
        "jenkins",
        "cicd",
        "infrastructure",
        "homelab",
        "selfhosted",
        "networking",
        "linuxquestions",
        "linux",
        "ubuntu",
        "centos",
        "redhat",
        "debian",
        "archlinux",
        "opnsense",
        "pfsense",
        "vmware",
        "proxmox",
        "unraid",
        "truenas",
        
        # Git & Version Control
        "git",
        "gitlab",
        "github",
        "bitbucket",
        "versioncontrol",
        "opensource",
        "FOSS",
        "freesoftware",
        "linuxmasterrace",
        
        # Security & DevSecOps
        "netsec",
        "cybersecurity",
        "infosec",
        "AskNetsec",
        "security",
        "devsecops",
        "appsec",
        "penetrationtesting",
        "redteam",
        "blueteam",
        "malware",
        "reverseengineering",
        "cryptography",
        "privacy",
        "privacytoolsIO",
        
        # Testing & Quality Assurance
        "QualityAssurance",
        "softwaretesting",
        "automation",
        "selenium",
        "testautomation",
        "unittest",
        "TDD",
        "BDD",
        
        # Agile & Project Management
        "agile",
        "scrum",
        "projectmanagement",
        "ProductManagement",
        "agilecoach",
        "kanban",
        "lean",
        
        # Startups & Business Tech
        "startups",
        "entrepreneur",
        "smallbusiness",
        "technology",
        "tech",
        "SaaS",
        "b2b",
        "EnterpriseIT",
        "business",
        "remotework",
        "digitalnomad",
        "WFH",
        
        # Specific Tools & Frameworks
        "docker",
        "kubernetes",
        "terraform",
        "ansible",
        "prometheus",
        "grafana",
        "elasticsearch",
        "mongodb",
        "postgresql",
        "mysql",
        "redis",
        "nginx",
        "apache",
        "helm",
        "istio",
        "envoy",
        "consul",
        "vault",
        
        # Mobile & Cross-platform
        "androiddev",
        "iOSProgramming",
        "reactnative",
        "flutter",
        "xamarin",
        "ionic",
        "mobiledev",
        
        # Data & Analytics
        "MachineLearning",
        "datascience",
        "analytics",
        "bigdata",
        "dataengineering",
        "MLOps",
        "artificial",
        "deeplearning",
        "tensorflow",
        "pytorch",
        
        # Game Development
        "gamedev",
        "Unity3D",
        "unrealengine",
        "godot",
        "IndieDev",
        
        # Design & UX
        "userexperience",
        "UI_Design",
        "web_design",
        "design",
        "graphic_design",
        
        # Specialized Communities
        "embedded",
        "arduino",
        "raspberry_pi",
        "IOT",
        "robotics",
        "3Dprinting",
        "maker",
        "electronics",
        "FPGA",
        
        # Learning & Career
        "cscareerquestions",
        "ITCareerQuestions",
        "LearnProgramming",
        "codereview",
        "AskProgramming",
        "ExperiencedDevs",
        "cscareerquestionsEU",
        "ITdept",
        "techsupport",
        "sysadminjobs",
        "devopsjobs",
        
        # Regional Tech Communities
        "developersindia",
        "cscareerquestionsEU",
        "UKTech",
        "CanadaTech",
        "GermanyInformatics",
        "EuropeanTech",
        "AsianTech"
    ]
    
    # Product-focused GitLab keywords (not financial)
    keywords = [
        "GitLab",
        "gitlab DevSecOps",
        "GitLab CI/CD",
        "GitLab pipeline",
        "GitLab runner",
        "gitlab-ci.yml",
        "GitLab DevOps",
        "GitLab merge request",
        "GitLab Repository",
        "GitLab Docker",
        "GitLab version control",
        "GitLab security scanning",
        "GitLab duo",
        "GitLab enterprise",
        "GitLab Code Quality",
        "GitLab Ultimate",
        "GitLab Dedicated",
        "GitLab Pages",
        "GitLab Container Registry",
        "GitLab Security Dashboard"
    ]
    
    # Parameters with removed minimum upvotes and updated end date
    min_upvotes = 1  # Removed minimum upvotes requirement
    min_comment_upvotes = 2  # Removed minimum comment upvotes requirement
    start_date = int(datetime(2025, 6, 6).timestamp())  
    end_date = int(datetime(2025, 7, 28).timestamp())  # Updated to today's date
    output_file = 'GTLB/GTLB2.csv'
    
    # Pull data
    df = pull_reddit_data(
        reddit=reddit,
        subreddit_names=subreddit_names,
        keywords=keywords,
        min_upvotes=min_upvotes,
        start_date=start_date,
        end_date=end_date,
        min_comment_upvotes=min_comment_upvotes
    )
    
    # Save to CSV
    save_to_csv(df, output_file)